# Ticari Orta Faz 1 Execution Simulation

Bu notebook mevcut pipeline'i dogrudan kullanarak asagidaki akisi simule etmek icin hazirlandi:

- Oracle native data kontrolu
- Native -> derived materialization
- Development / training
- Calibration
- Supervised validation / outcome evaluation
- Weight tuning denemesi
- Promote
- Single snapshot scoring
- Date range scoring
- Monitoring ve Oracle output inceleme
- Derived tabloda DATA_TIME kontrolu

Notebook formulleri yeniden yazmaz; dogrudan repo icindeki `LifecycleManager`, `NativeMaterializer` ve `OracleConnector` siniflarini kullanir.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import display

from engine.config_loader import load_config, load_secrets
from engine.lifecycle import LifecycleManager
from engine.materialization import NativeMaterializer
from engine.oracle_io import OracleConnector
from engine.ticari_orta_faz1_demo import TicariOrtaFaz1DemoBuilder

PROJECT_ROOT = Path.cwd()
config = load_config()
secrets = load_secrets()
manager = LifecycleManager()
materializer = NativeMaterializer(config, secrets)

SEGMENT = "TICARI_ORTA_FAZ1"
SINGLE_SNAPSHOT = "2026-04-30"
RANGE_START = "2026-01-31"
RANGE_END = "2026-04-30"
USE_SYNTHETIC_NATIVE_IF_EMPTY = False


In [ ]:
def compact_dict(payload):
    if not isinstance(payload, dict):
        return payload
    return {key: value for key, value in payload.items() if key != "frame"}

def safe_run(label, fn, *args, **kwargs):
    try:
        result = fn(*args, **kwargs)
        print(f"{label}: OK")
        return result
    except Exception as exc:
        print(f"{label}: FAILED -> {exc}")
        return {"error": str(exc)}

overview = {
    "pipeline_name": config["pipeline"]["name"],
    "segment": SEGMENT,
    "native_source": config["sources"]["native_features"]["oracle"]["table"],
    "derived_source": config["sources"]["input_features"]["oracle"]["table"],
    "outcomes_source": config["sources"]["outcomes"]["oracle"]["table"],
    "score_live_selector": config["live_scoring"]["snapshot"]["selector"],
    "materialization_enabled": config["materialization"]["enabled"],
    "warmup_snapshots": config["materialization"]["trim_warmup_snapshots"],
}
display(pd.DataFrame([overview]))


## 1. Optional Synthetic Native Seed

Native tablo henuz dolu degilse bu hucreyi `True` yapip ayni notebook icinden sentetik native veri basabilirsin.

- `False`: mevcut Oracle native tabloyu kullan
- `True`: sentetik native + outcomes uret, sonra kalan notebook'u calistir


In [ ]:
if USE_SYNTHETIC_NATIVE_IF_EMPTY:
    demo_builder = TicariOrtaFaz1DemoBuilder()
    synthetic_prepare = safe_run(
        "synthetic_prepare",
        demo_builder.prepare,
        segment=SEGMENT,
        num_customers=180,
        drop_existing=False,
    )
    display(pd.DataFrame([compact_dict(synthetic_prepare)]))
else:
    print("Sentetik native seed kapali; mevcut Oracle native tablo kullanilacak.")


## 1. Native Data Checks

Native tablo mevcut mu, hangi tarih araligini kapsiyor ve secilen segmentte kac satir var gorelim.

In [ ]:
with OracleConnector(config, secrets) as ora:
    native_table = ora._qualified_table_name("native_features")
    native_overview = ora._read_query(
        f"""
        SELECT
            MIN(SNAPSHOT_DATE) AS min_snapshot_date,
            MAX(SNAPSHOT_DATE) AS max_snapshot_date,
            COUNT(*) AS row_count,
            COUNT(DISTINCT CUSTOMER_ID) AS customer_count
        FROM {native_table}
        WHERE SEGMENT = :segment_value
        """,
        {"segment_value": SEGMENT},
    )
    native_sample = ora._read_query(
        f"""
        SELECT *
        FROM {native_table}
        WHERE SEGMENT = :segment_value
          AND ROWNUM <= 5
        ORDER BY SNAPSHOT_DATE DESC
        """,
        {"segment_value": SEGMENT},
    )

display(native_overview)
display(native_sample)


## 2. Materialization

Ayni kod yolunu kullanarak native veriden derived/input tablosu olusturuyoruz.

- `materialize_development` full history rebuild
- `materialize_live` ise tek snapshot veya date range icin scope refresh

In [ ]:
full_materialization = safe_run(
    "materialize_development",
    materializer.materialize_development,
    SEGMENT,
)
display(pd.DataFrame([compact_dict(full_materialization)]))


In [ ]:
single_materialization = safe_run(
    "materialize_live(single_snapshot)",
    materializer.materialize_live,
    SEGMENT,
    snapshot_date=SINGLE_SNAPSHOT,
)

display(pd.DataFrame([compact_dict(single_materialization)]))
if isinstance(single_materialization, dict) and isinstance(single_materialization.get("frame"), pd.DataFrame):
    display(single_materialization["frame"].head())


## 3. Development / Training / Calibration

Bu adim modeli fit eder, calibration artifact uretir ve registry'ye candidate model olarak yazar.

In [ ]:
develop_summary = safe_run("develop", manager.develop, segment=SEGMENT)
display(pd.DataFrame([develop_summary]))


## 4. Supervised Validation

Outcome tablosu varsa supervised outcome evaluation ve weight tuning denenebilir.

In [ ]:
evaluation_summary = safe_run(
    "evaluate_outcomes",
    manager.evaluate_outcomes,
    segment=SEGMENT,
    model_version=develop_summary.get("model_version") if isinstance(develop_summary, dict) else None,
)
display(pd.DataFrame([evaluation_summary]))

weight_tuning_summary = safe_run(
    "tune_weights",
    manager.tune_weights,
    segment=SEGMENT,
    model_version=develop_summary.get("model_version") if isinstance(develop_summary, dict) else None,
    apply=False,
)
display(pd.DataFrame([weight_tuning_summary]))


## 5. Promote and Live Scoring

Candidate model champion olarak promote edilir ve hem single snapshot hem de date range scoring cagrilari yapilir.

In [ ]:
promote_summary = safe_run(
    "promote",
    manager.promote,
    segment=SEGMENT,
    model_version=develop_summary.get("model_version") if isinstance(develop_summary, dict) else None,
)
display(pd.DataFrame([promote_summary]))

single_score_summary = safe_run(
    "score_live(single_snapshot)",
    manager.score_live,
    segment=SEGMENT,
    snapshot_date=SINGLE_SNAPSHOT,
)
display(pd.DataFrame([single_score_summary]))

range_score_summary = safe_run(
    "score_live(range)",
    manager.score_live,
    segment=SEGMENT,
    start_date=RANGE_START,
    end_date=RANGE_END,
)
display(pd.DataFrame([range_score_summary]))


## 6. Monitoring and Oracle Output Review

Skor ciktisinin sonuclarini ve monitoring JSON ozetini okuyalim.

In [ ]:
with OracleConnector(config, secrets) as ora:
    results_table = ora._qualified_table_name("results")
    details_table = ora._qualified_table_name("details")
    effects_table = ora._qualified_table_name("full_effects")
    input_table = ora._qualified_table_name("input_features")

    result_sample = ora._read_query(
        f"""
        SELECT CUSTOMER_ID, SNAPSHOT_DATE, ANOMALY_SCORE, ALERT_BAND, REASON_1, REASON_2, REASON_3
        FROM {results_table}
        WHERE TRUNC(SNAPSHOT_DATE) = DATE '{SINGLE_SNAPSHOT}'
          AND ROWNUM <= 5
        ORDER BY ANOMALY_SCORE DESC
        """
    )
    details_sample = ora._read_query(
        f"""
        SELECT CUSTOMER_ID, SNAPSHOT_DATE, FEATURE_NAME, FEATURE_LABEL,
               ACTUAL_VALUE, CUSTOMER_HISTORY_REFERENCE, POPULATION_REFERENCE,
               AE_REFERENCE, CONTRIBUTION_PCT, AE_CONTRIBUTION_PCT, IF_CONTRIBUTION_PCT, MD_CONTRIBUTION_PCT
        FROM {details_table}
        WHERE TRUNC(SNAPSHOT_DATE) = DATE '{SINGLE_SNAPSHOT}'
          AND ROWNUM <= 10
        ORDER BY FEATURE_RANK ASC
        """
    )
    data_time_check = ora._read_query(
        f"""
        SELECT TRUNC(SNAPSHOT_DATE) AS snapshot_date,
               COUNT(*) AS row_count,
               MIN(DATA_TIME) AS min_data_time,
               MAX(DATA_TIME) AS max_data_time
        FROM {input_table}
        WHERE TRUNC(SNAPSHOT_DATE) = DATE '{SINGLE_SNAPSHOT}'
        GROUP BY TRUNC(SNAPSHOT_DATE)
        """
    )

display(result_sample)
display(details_sample)
display(data_time_check)


In [ ]:
monitoring_path = single_score_summary.get("monitoring_path") if isinstance(single_score_summary, dict) else None
if monitoring_path:
    monitoring_payload = json.loads(Path(monitoring_path).read_text(encoding="utf-8"))
    display(pd.DataFrame([monitoring_payload.get("scores", {})]))
    display(pd.DataFrame([monitoring_payload.get("input", {})]))
else:
    print("Monitoring payload bulunamadi.")


## 7. Notes

- Native tablo ayni contract ile geldiginde notebook tekrar kullanilabilir.
- Derived/input rematerialization score cagrisi icinde otomatik yapilir.
- Single-date score ayni snapshot icin tekrar kosarsa input tablosundaki `DATA_TIME` guncellenir.
- Range scoring output tablolarinda ilgili tarih araligini replace eder.
- Scheduler bu notebook icinden degil, dis orchestration araci uzerinden tetiklenmelidir.